# Cloud reader: Qwen2.5-14B-Instruct

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When the **Choose Files** button appears below, pick `cloud/prompts.jsonl` from the project folder
4. Wait (about 30–45 min). `answers.json` downloads automatically at the end

In [ ]:
!pip -q install -U bitsandbytes accelerate

In [ ]:
import json, re, torch
from google.colab import files
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
uploaded = files.upload()
rows = [json.loads(line) for line in open(next(iter(uploaded)), encoding="utf-8")]
print(len(rows), "prompts")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL = "Qwen/Qwen2.5-14B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16))
model.eval()
print("loaded", torch.cuda.memory_allocated() / 1e9, "GB")

In [ ]:
from tqdm.auto import tqdm
REFUSALS = re.compile(r"\b(unknown|not (mentioned|provided|specified|available|listed|stated|found|given)|n/a|tbd|tba|"
                      r"no information|cannot be determined|does not (say|mention|contain|specify))\b", re.I)

def is_refusal(answer):
    text = answer.strip(" .").lower()
    return not text or text == "none" or bool(REFUSALS.search(text))

@torch.no_grad()
def generate(messages, max_tokens):
    prompts = [tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in messages]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [t.strip().split("\n")[0].strip() for t in texts]

answers = {}
for attempt, batch_size in enumerate([8, 4, 16]):
    todo = [r for r in rows if attempt == 0 or is_refusal(answers[r["key"]])]
    todo.sort(key=lambda r: (r["max_tokens"], len(json.dumps(r["attempts"][attempt]))))
    for i in tqdm(range(0, len(todo), batch_size), desc=f"attempt {attempt + 1}: {len(todo)} prompts"):
        batch = todo[i:i + batch_size]
        for r, a in zip(batch, generate([r["attempts"][attempt] for r in batch], max(r["max_tokens"] for r in batch))):
            answers[r["key"]] = a
    json.dump(answers, open("answers.json", "w"), ensure_ascii=False)
print("refusals left:", sum(is_refusal(a) for a in answers.values()))

In [ ]:
files.download("answers.json")